### 导包

In [ ]:
import pandas as pd
import numpy as np
import ta
from ta import add_all_ta_features
from ta.utils import dropna
import finrl
from finrl.meta.preprocessor.preprocessors import data_split
from finrl.meta.preprocessor.preprocessors import FeatureEngineer
from sklearn.preprocessing import RobustScaler
import lightgbm as lgb
from sklearn.metrics import r2_score

In [ ]:
data_with_features = pd.read_csv("../datasets/S&P500_feature.csv")

In [ ]:
list_American = data_with_features.tic.unique()

In [ ]:
data_with_features.columns[2:]

In [ ]:
data_with_features.head()

In [ ]:
def time_decay(timestamps, half_life=-150):
    current_time = pd.Timestamp("2023-01-01")
    weights = []
    for ts in timestamps:
        ts_timestamp = pd.Timestamp(ts)
        age = (current_time - ts_timestamp).days
        weight =  np.exp(-age / half_life)
        weights.append(weight)
    return weights

In [ ]:
pre_return = pd.DataFrame()
list_return = []
for i in range(len(data_with_features.tic.unique())):
    df = data_with_features[data_with_features["tic"] == list_American[i]]
    df = df[(df.date >= "2018-01-01") & (df.date < "2024-01-01")]
    df.fillna(0, inplace=True)
    y = df["close"].pct_change()
    y = y.shift(-1)  # 预测次日收益率
    y = y[:-1]  # 对齐数据
    # print(y)
    df = df.set_index("date")
    df = df[df.columns[6:] ]
    # 数据清洗
    scaler = RobustScaler()
    X_train = scaler.fit_transform(df[(df.index < "2023-01-01") & (df.index >= "2018-01-01")])
    X_test = scaler.fit_transform(df[(df.index >= "2023-01-01") & (df.index < "2024-01-01")])
    # print(len(X_train))
    # print(len(X_test))
    X_test = X_test[:-1]
    y_train, y_test = y[:len(X_train)], y[len(X_train):]
    # train_timestamps = df[(df.index < "2023-01-01") & (df.index >= "2017-01-01")].index
    # train_weights = time_decay(train_timestamps)
    model = lgb.LGBMRegressor(
        num_leaves=12, 
        n_estimators=500, 
        reg_lambda=2, 
        learning_rate=0.001
        )
    
    # model = lgb.LGBMRegressor(
    #     num_leaves=20, 
    #     n_estimators=200, 
    #     reg_lambda=5, 
    #     learning_rate=0.0001
    # )
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    reward = (1 + pred).cumprod()
    pre_return[i] = reward
    print(f"R² Score: {r2_score(y_test, pred):.3f}")

In [ ]:
pre_return

In [ ]:
pre_return = pre_return[-1:].T

In [ ]:
pre_return = pre_return.set_index(list_American)

In [ ]:
pre_return.columns  = ["pre_reward"]

In [ ]:
pre_return = pre_return.sort_values("pre_reward",ascending=False)

In [ ]:
pre_return.head(20)

In [ ]:
list_American_top = pre_return.head(20).index

In [ ]:
list_American_top

In [ ]:
data_American_selection = data_with_features[data_with_features.tic.isin(list_American_top)]

In [ ]:
data_American_selection = data_American_selection[(data_American_selection.date >= "2017-01-01") & (data_American_selection.date < "2024-01-01")]

In [ ]:
data_American_selection

In [ ]:
data_American_selection.to_csv("datasets/stock20_Feature.csv",index=False)

In [ ]:
data = pd.read_csv("../datasets/S&P500.csv")

In [ ]:
data

In [ ]:
data_American_selection1 = data[data.tic.isin(list_American_top)]

In [ ]:
data_American_selection1 = data_American_selection1[(data_American_selection1.date >= "2017-01-01") & (data_American_selection1.date < "2024-01-01")]

In [ ]:
data_American_selection1

In [ ]:
data_American_selection1.to_csv("datasets/stock20_Basic.csv",index=False)

In [ ]:
test_return = pd.DataFrame()
for i in range(len(data_with_features.tic.unique())):
    df = data_with_features[data_with_features["tic"] == list_American[i]]
    df = df[df.date < "2024-01-01"]
    df.fillna(0, inplace=True)
    y = df["close"].pct_change()
    y = y.shift(-1)  # 预测次日收益率
    y = y[:-1]  # 对齐数据
    # print(y)
    df = df.set_index("date")
    df = df[df.columns[2:] ]
    # 数据清洗
    scaler = RobustScaler()
    X_train = scaler.fit_transform(df[(df.index < "2023-01-01") & (df.index >= "2019-01-01")])
    X_test = scaler.fit_transform(df[df.index >= "2023-01-01"])
    # print(len(X_train))
    # print(len(X_test))
    X_test = X_test[:-1]
    y_train, y_test = y[:len(X_train)], y[len(X_train):]
    # train_timestamps = df[(df.index < "2023-01-01") & (df.index >= "2017-01-01")].index
    # train_weights = time_decay(train_timestamps)
    # X_test = X_test[:-1]
    # y_train, y_test = y[:len(X_train)], y[len(X_train):]
    y_test = y_test.reset_index()
    y_test = y_test.drop("index",axis=1)
    reward = (1+ y_test).cumprod()
    test_return[i] = reward
    print("stock_name:",list_American[i],"return",reward)

In [ ]:
test_return = test_return[-1:].T
test_return = test_return.set_index(list_American)
test_return.columns = ["reward"]
test_return = test_return.sort_values("reward",ascending=False)
test_return.head(20)